# Frozen seven-model test comparison

Run this notebook with a Colab GPU. It evaluates five already-selected models on the same 960-row reused held-out split. It does not select parameters, promote a model, or claim a new blind test.

## Setup

The branch and all external model revisions are pinned. Re-running is safe because embeddings are cached in Google Drive when mounted.

In [ ]:
import os

os.chdir("/content")

REPOSITORY = "https://github.com/ylceadap/sentiment-analysis.git"
BRANCH = "main"
WORKSPACE = "/content/sentiment-analysis"
DRIVE_CACHE = "/content/dutch-sentiment-jina-cache"

In [ ]:
%cd /content
!rm -rf {WORKSPACE}
!git clone --depth 1 --branch {BRANCH} {REPOSITORY} {WORKSPACE}
%cd {WORKSPACE}
!git fetch -q origin refs/tags/archive/ordinal-logistic/2026-07-19:refs/tags/archive/ordinal-logistic/2026-07-19
!git fetch -q origin refs/tags/archive/llm/2026-07-19:refs/tags/archive/llm/2026-07-19
# Colab currently uses Python 3.12; the flag bypasses only the package metadata gate.
!python -m pip install -q --ignore-requires-python -e '.[train,embeddings]'

## Run and validate

This cell redirects only ignored caches to Drive. Outputs remain in the cloned repository so they can be downloaded as one archive.

In [ ]:
import os
from pathlib import Path

import yaml

Path(DRIVE_CACHE).mkdir(parents=True, exist_ok=True)
config_path = Path("configs/final_model_comparison.yaml")
config = yaml.safe_load(config_path.read_text())
config["jina"]["cache_dir"] = f"{DRIVE_CACHE}/embeddings"
config["jina"]["huggingface_cache_dir"] = f"{DRIVE_CACHE}/huggingface"
config["jina"]["batch_size"] = 16
config["jina"]["cache_shard_size"] = 256
runtime_config = Path("/content/final_model_comparison_colab.yaml")
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
os.environ["HF_MODULES_CACHE"] = f"{DRIVE_CACHE}/modules"
os.environ["HF_HUB_CACHE"] = f"{DRIVE_CACHE}/huggingface"
!python -m dutch_sentiment.final_comparison --config {runtime_config}

In [ ]:
import json

import pandas as pd

result = json.loads(Path("artifacts/final_models/comparison.json").read_text())
assert result["evaluation_scope"] == "reused-heldout-presentation-comparison"
assert result["data"]["heldout_rows"] == 960
assert len(result["ranking"]) == 5
pd.DataFrame(result["ranking"])

## Export

Download the zip and place its contents back into the repository. The local `make final-compare` command will then hit the verified embedding cache and log the complete result to MLflow.

In [ ]:
from google.colab import files

!zip -qr /content/final_model_results.zip artifacts/final_models reports/final_model_comparison.md
files.download("/content/final_model_results.zip")